In [1]:
import sys
print(sys.executable)

c:\Users\vijay\rag-project\venv\Scripts\python.exe


In [2]:
import fitz  # PyMuPDF

def load_pdf(path):
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages

pages = load_pdf("../data/doc.pdf")
print(f"Loaded {len(pages)} pages")
print(pages[0]["text"][:500])

c:\Users\vijay\rag-project\venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Loaded 15 pages
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [3]:
def chunk_pages(pages, chunk_size=150, overlap=30, source="doc.pdf"):
    chunks = []
    step = chunk_size - overlap
    for p in pages:
        words = p["text"].split()
        for start in range(0, len(words), step):
            piece = words[start:start + chunk_size]
            if len(piece) < 20:   # skip tiny leftovers
                continue
            chunks.append({
                "text": " ".join(piece),
                "page": p["page"],
                "source": source,
            })
    return chunks

chunks = chunk_pages(pages)
print(f"Created {len(chunks)} chunks")
print(chunks[5])

Created 58 chunks
{'text': 'at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains. Attention mechanisms have become an integral part of compelling sequence modeling and transduc- tion models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("Vectors in index:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (58, 384)
Vectors in index: 58


In [7]:
def search(query, k=3):
    q = model.encode([query], normalize_embeddings=True, show_progress_bar=False)
    q = np.array(q, dtype="float32")
    scores, ids = index.search(q, k)
    results = []
    for score, i in zip(scores[0], ids[0]):
        results.append({"score": float(score), **chunks[i]})
    return results

import textwrap

def show(query, k=5):
    print(f"\nQUESTION: {query}\n" + "=" * 70)
    for rank, r in enumerate(search(query, k), 1):
        print(f"#{rank} [score {r['score']:.3f} | page {r['page']}]")
        print(textwrap.fill(r["text"], width=100), "\n")

show("What is self-attention?")
show("Why is attention better than recurrence?")
show("How many attention heads does the base model use?")
show("What optimizer was used for training?")


QUESTION: What is self-attention?
#1 [score 0.515 | page 6]
easily learn to attend by relative positions, since for any fixed offset k, PEpos+k can be
represented as a linear function of PEpos. We also experimented with using learned positional
embeddings [9] instead, and found that the two versions produced nearly identical results (see Table
3 row (E)). We chose the sinusoidal version because it may allow the model to extrapolate to
sequence lengths longer than the ones encountered during training. 4 Why Self-Attention In this
section we compare various aspects of self-attention layers to the recurrent and convolu- tional
layers commonly used for mapping one variable-length sequence of symbol representations (x1, ...,
xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈Rd, such as a hidden layer in a
typical sequence transduction encoder or decoder. Motivating our use of self-attention we consider
three desiderata. One is the total computational complexity per layer.

In [8]:
import ollama

response = ollama.chat(
    model="qwen2.5:1.5b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(response["message"]["content"])

Hello! How can I assist you today?


In [9]:
import fitz, numpy as np, faiss, textwrap
from sentence_transformers import SentenceTransformer

def load_pdf(path):
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages

def chunk_pages(pages, chunk_size=150, overlap=30, source="doc.pdf"):
    chunks = []
    step = chunk_size - overlap
    for p in pages:
        words = p["text"].split()
        for start in range(0, len(words), step):
            piece = words[start:start + chunk_size]
            if len(piece) < 20:
                continue
            chunks.append({"text": " ".join(piece), "page": p["page"], "source": source})
    return chunks

pages = load_pdf("../data/doc.pdf")
chunks = chunk_pages(pages)

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def search(query, k=3):
    q = model.encode([query], normalize_embeddings=True, show_progress_bar=False)
    q = np.array(q, dtype="float32")
    scores, ids = index.search(q, k)
    return [{"score": float(s), **chunks[i]} for s, i in zip(scores[0], ids[0])]

def show(query, k=5):
    print(f"\nQUESTION: {query}\n" + "=" * 70)
    for rank, r in enumerate(search(query, k), 1):
        print(f"#{rank} [score {r['score']:.3f} | page {r['page']}]")
        print(textwrap.fill(r["text"], width=100), "\n")

print(f"{len(pages)} pages, {len(chunks)} chunks, {index.ntotal} vectors in index")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

15 pages, 58 chunks, 58 vectors in index


In [16]:
def answer(question, k=4, model_name="qwen2.5:1.5b"):
    results = search(question, k)

    context = "\n\n".join(
        f"[Source {i+1}, page {r['page']}]\n{r['text']}"
        for i, r in enumerate(results)
    )

    prompt = f"""Answer the question using ONLY the context below.

Rules:
- Write 2 to 4 complete sentences.
- After each sentence, add the source tag it came from, like [Source 2].
- If the context does not contain the answer, reply exactly: I could not find this in the document.

Example of the format (unrelated topic):
Water boils at 100 degrees Celsius at sea level. [Source 2] The boiling point drops at higher altitudes. [Source 1]
CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"], results

text, sources = answer("What is self-attention?")
print(text)
print("\nPages retrieved:", [s["page"] for s in sources])

Self-attention is a mechanism used in the Transformer model, where each position in the sequence can attend to all positions in the input sequence and can also attend to all positions in the previous layer of the encoder. This allows every position in the decoder to attend over all positions in the input sequence, mimicking the typical encoder-decoder attention mechanisms in sequence-to-sequence models.

Pages retrieved: [6, 5, 3, 14]


In [17]:
def ask(q, k=4):
    text, sources = answer(q, k=k)
    print("Q:", q)
    print(textwrap.fill(text, width=100))
    print("Pages retrieved:", [s["page"] for s in sources])
    print("-" * 100)

for q in [
    "What is self-attention?",
    "Why is attention better than recurrence?",
    "How many attention heads does the base model use?",
    "What optimizer was used for training?",
    "Who won the 2018 World Cup?",
]:
    ask(q)

Q: What is self-attention?
Self-attention is a mechanism used in the Transformer model to allow a position in the sequence to
attend to all positions in the input sequence. This is achieved through multi-head attention, where
each position can attend to all positions in the previous layer of the encoder.
Pages retrieved: [6, 5, 3, 14]
----------------------------------------------------------------------------------------------------
Q: Why is attention better than recurrence?
Attention mechanisms allow for significantly more parallelization and can reach a new state of the
art in translation quality after being trained. [Source 4, page 2]
Pages retrieved: [6, 5, 3, 2]
----------------------------------------------------------------------------------------------------
Q: How many attention heads does the base model use?
The base model uses 8 attention heads. [Source 3, page 5]
Pages retrieved: [14, 5, 5, 15]
------------------------------------------------------------------------------